In [1]:
import pandas as pd

In [2]:
def set_bit(mask, i):
    bit = 1 << i
    return mask | bit

def clear_bit(mask, i):
    bit = 1 << i
    return ~bit & mask

def flip_bit(mask, i):
    bit = 1 << i
    return mask ^ bit

def get_bit(mask, i):
    bit = 1 << i
    return bool(mask & bit)


In [3]:
df = pd.read_csv('db/title.basics.tsv', sep='\t', na_values='\\N')

C:\Users\muniz\AppData\Local\Temp\ipykernel_22608\2934157030.py:1: DtypeWarning: Columns (0: runtimeMinutes) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('db/title.basics.tsv', sep='\t', na_values='\\N')


In [4]:
df = df[df['titleType'] == 'movie']
df = df[df['runtimeMinutes'].notna()]
df['runtimeMinutes'] = df['runtimeMinutes'].astype(int)
df.drop(columns=['titleType'], inplace=True)

In [5]:
df.drop(columns=['endYear'], inplace=True)

In [6]:
df['isAdult'] = df['isAdult'] == 1

In [7]:
df.info()

<class 'pandas.DataFrame'>
Index: 468005 entries, 8 to 12402706
Data columns (total 7 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   tconst          468005 non-null  str    
 1   primaryTitle    468004 non-null  str    
 2   originalTitle   468004 non-null  str    
 3   isAdult         468005 non-null  bool   
 4   startYear       460052 non-null  float64
 5   runtimeMinutes  468005 non-null  int64  
 6   genres          441550 non-null  str    
dtypes: bool(1), float64(1), int64(1), str(4)
memory usage: 51.2 MB


In [8]:
df.dropna(subset=['genres'], inplace=True)

In [9]:
from functools import reduce

df['genres'] = df['genres'].apply(lambda l: l.split(','))
genres = list(reduce(lambda acc, x: set(acc) | set(x), df['genres'], set()))
print(genres)

['Sci-Fi', 'Biography', 'Crime', 'War', 'Film-Noir', 'Western', 'Talk-Show', 'Comedy', 'Adult', 'Family', 'History', 'Fantasy', 'Action', 'Animation', 'Documentary', 'Game-Show', 'Thriller', 'Mystery', 'Reality-TV', 'News', 'Horror', 'Adventure', 'Sport', 'Music', 'Romance', 'Musical', 'Drama']


In [10]:
genres_by_name = {}
for i, genre in enumerate(genres):
    genres_by_name[genre] = i

print(genres_by_name)


{'Sci-Fi': 0, 'Biography': 1, 'Crime': 2, 'War': 3, 'Film-Noir': 4, 'Western': 5, 'Talk-Show': 6, 'Comedy': 7, 'Adult': 8, 'Family': 9, 'History': 10, 'Fantasy': 11, 'Action': 12, 'Animation': 13, 'Documentary': 14, 'Game-Show': 15, 'Thriller': 16, 'Mystery': 17, 'Reality-TV': 18, 'News': 19, 'Horror': 20, 'Adventure': 21, 'Sport': 22, 'Music': 23, 'Romance': 24, 'Musical': 25, 'Drama': 26}


In [11]:
def genre_list_to_bitmask(genre_list: list[str]) -> int:
    mask = 0b0
    for genre in genre_list:
        i = genres_by_name[genre]
        set_bit(mask, i)
    return mask

In [12]:
df['genres'] = df['genres'].apply(genre_list_to_bitmask)

In [13]:
df.fillna(None, inplace=True)
df.to_parquet('db/title.basics.parquet')